In [2]:
from dotenv import load_dotenv
from huggingface_hub import login
import torch
from transformers import pipeline
from datasets import load_dataset
from tqdm import tqdm
import os

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cuda


In [3]:
import sys
print(sys.executable)

import ipywidgets
print(ipywidgets.__version__)

/work/Folder/Llava-Adapter/.venv/bin/python
8.1.9


In [4]:
# Test run
print(torch.backends.mps.is_available())

pipe = pipeline(
    "image-text-to-text",
    model="llava-hf/llava-1.5-7b-hf",
    device=device,
    dtype=torch.float16,
)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"},
        {"type": "text", "text": "What does the label 15 represent? (1) lava (2) core (3) tunnel (4) ash cloud"},
    ],
}]

out = pipe(text=messages, max_new_tokens=20)
print(out[0]["generated_text"])

False


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user', 'content': [{'type': 'image', 'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg'}, {'type': 'text', 'text': 'What does the label 15 represent? (1) lava (2) core (3) tunnel (4) ash cloud'}]}, {'role': 'assistant', 'content': ' The label 15 represents the ash cloud, which is a natural phenomenon associated with volcan'}]


In [ ]:
# Load in the chess pieces dataset
# Simple toy dataset with 50 images to test learning capability

dataset = load_dataset('Trelis/chess_pieces')
print(dataset)
print(dataset["test"].features) 

def make_messages(img):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Identify the type chess piece/s in the image as well as their colour"},
        ],
    }]

def run(row):
    return pipe(text=make_messages(row["image"]), max_new_tokens=60,
                return_full_text=False)[0]["generated_text"]

run(dataset["test"][0])

for i in tqdm(range(len(dataset["test"]))):
    row = dataset["test"][i]
    tqdm.write(run(row))


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DatasetDict({
    train: Dataset({
        features: ['image', 'caption'],
        num_rows: 48
    })
    test: Dataset({
        features: ['image', 'caption'],
        num_rows: 3
    })
})
{'image': Image(mode=None, decode=True), 'caption': Value('string')}


 33%|███▎      | 1/3 [00:00<00:00,  2.48it/s][transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 In the image, there is a white chess piece, which is a knight, and a brown chess piece, which is a rook.


 67%|██████▋   | 2/3 [00:00<00:00,  2.66it/s][transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 In the image, there are two chess pieces, one is a white knight and the other is a black knight.


100%|██████████| 3/3 [00:01<00:00,  2.56it/s]

 In the image, there is a white chess piece, which is a knight, and a black chess piece, which is a black knight.


In [20]:
# Run the model on a harder task
# CUB-200 Birds dataset

dataset = load_dataset("cassiekang/cub200_dataset")
print(dataset)
print(dataset["test"].features) 

def make_messages(img):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Identify the bird in the photo"},
        ],
    }]

def run(row):
    return pipe(text=make_messages(row["image"]), max_new_tokens=60,
                return_full_text=False)[0]["generated_text"]

run(dataset["test"][0])

for i in tqdm(range(3)):
    row = dataset["train"][i]
    pred = run(row)
    tqdm.write(f"[{i}] Pred:  {pred.strip()}\n[{i}] Truth: {row['text']}\n")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DatasetDict({
    train: Dataset({
        features: ['text', 'image'],
        num_rows: 5994
    })
    test: Dataset({
        features: ['text', 'image'],
        num_rows: 5794
    })
})
{'text': Value('string'), 'image': Image(mode=None, decode=True)}


 67%|██████▋   | 2/3 [00:00<00:00,  5.92it/s]

[0] Pred:  The bird in the photo is a seagull.
[0] Truth: A photo of a laysan albatross

[1] Pred:  The bird in the photo is a small, gray and white bird, possibly a sparrow.
[1] Truth: A photo of a sage thrasher



[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 3/3 [00:00<00:00,  5.82it/s]

[2] Pred:  The bird in the photo is a small brown and white bird, possibly a sparrow.
[2] Truth: A photo of a bank swallow

